In [ ]:
import matplotlib.pyplot as plt

from geo_data import data_handler, helpers, map_style
from geo_data.projections import Equirectangular
from geo_data.svg_handler import MapSVG

# TODO:
# das hier kann man vielleicht benutzen, um es nicht ganz so detailreich zu machen:
# germany["geometry"] = germany["geometry"].simplify(0.01, preserve_topology=True)

# Load data

In [ ]:
countries = data_handler.load(kind="country", source="ne", resolution=10)
all_states = data_handler.load(kind="state", source="ne", resolution=10)
all_rivers = data_handler.load(
    kind="river", source="ne", resolution=10, identifier="name_de"
)

germany = countries[countries["name"] == "Germany"]
states = all_states[all_states["admin"] == "Germany"]
rivers = all_rivers.clip(germany)

# Plot data and create an SVG-file with matplotlib

In [ ]:
fig, ax = plt.subplots()
fig.set_facecolor(map_style.COLORS["background"])
ax.set_axis_off()

# boundaries
germany.plot(ax=ax, color=map_style.COLORS["land"], zorder=1)
germany.boundary.plot(ax=ax, color=map_style.COLORS["border"], linewidth=1.2, zorder=2)
states.boundary.plot(ax=ax, color=map_style.COLORS["border"], linewidth=0.4, zorder=3)
rivers.plot(ax=ax, color=map_style.COLORS["river"], linewidth=0.5, zorder=4)

result_path = helpers.get_top_directory() / "results" / "Germany_location_map_plt.svg"
plt.savefig(result_path, format="svg", bbox_inches="tight")

# Create SVG-file with the svg_handler

In [ ]:
geom_germany = germany.iloc[0].geometry
# bounds used by Wikipedia (in lon/lat)
bounds = (5.5, 47.2, 15.5, 55.1)

# Wikipedia uses an equirectangular projection (plate) with N/S stretching 150 %
projection = Equirectangular(y_scale=1.5)

canvas = MapSVG(bounds=bounds, width=1000, projection=projection)
canvas.add_background()

# german outer bounds
outer_bound_kwargs = dict(map_style.STYLES["land"])
outer_bound_kwargs["stroke_width"] = 3
canvas.add_gdf(germany, "Countries", **outer_bound_kwargs)

# state bounds
canvas.add_gdf(states, "States", **map_style.STYLES["land"])

# rivers
canvas.add_gdf(rivers, "Rivers", **map_style.STYLES["river"])

result_path = helpers.get_top_directory() / "results" / "Germany_location_map.svg"
canvas.save(result_path)
canvas